# The Judge — Bias & Fairness in AI
### Predicting Income with the Adult Census Dataset

This notebook builds a classifier to predict whether an individual earns more than $50K/year, then audits the model for fairness across **sex** and **race**.

## Step 1: Load the Data

In [2]:
import pandas as pd

df = pd.read_csv('adult.csv')
print('Shape:', df.shape)
df.head()

Shape: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## Step 2: Clean the Data

The dataset uses `'?'` as a placeholder for missing values (instead of true NaNs) in `workclass`, `occupation`, and `native.country`. We convert these to real NaNs and drop the affected rows, since they make up less than 8% of the data and we still have plenty of rows left after dropping.

In [3]:
# Strip whitespace from string columns (common quirk in this dataset)
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# Replace '?' placeholders with real NaN
df = df.replace('?', pd.NA)
print('Missing values by column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df_clean = df.dropna().reset_index(drop=True)
print(f'\nDropped {len(df) - len(df_clean)} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')
print('Clean shape:', df_clean.shape)

Missing values by column:
workclass         1836
occupation        1843
native.country     583
dtype: int64

Dropped 2399 rows (7.4%)
Clean shape: (30162, 15)


## Step 3: Explore Income Disparities (Why Fairness Matters Here)

Before modeling, we check the base rate of high income across sex and race groups. These gaps are a signal our model could learn to replicate.

In [4]:
print('Income >50K rate by sex:')
print(df_clean.groupby('sex')['income'].apply(lambda x: (x=='>50K').mean()))

print('\nIncome >50K rate by race:')
print(df_clean.groupby('race')['income'].apply(lambda x: (x=='>50K').mean()))

Income >50K rate by sex:
sex
Female    0.113678
Male      0.313837
Name: income, dtype: float64

Income >50K rate by race:
race
Amer-Indian-Eskimo    0.118881
Asian-Pac-Islander    0.277095
Black                 0.129925
Other                 0.090909
White                 0.263718
Name: income, dtype: float64


## Step 4: Encode Features

**Key design decision:** `sex` and `race` are removed from the model's training features and held aside separately. We do NOT want the model directly using these attributes to predict income — but we keep them so we can audit the model's predictions for fairness afterward.

We also drop:
- `fnlwgt` — a census sampling weight, not a real predictive feature
- `education` — redundant, since `education.num` already encodes the same information numerically

Remaining categorical columns are one-hot encoded.

In [5]:
from sklearn.model_selection import train_test_split

# Target: binary encode income
y = (df_clean['income'] == '>50K').astype(int)

# Sensitive attributes - held aside for the fairness audit
sensitive = df_clean[['sex', 'race']].copy()

# Features - drop target, fnlwgt, education, and sensitive attributes
X = df_clean.drop(columns=['income', 'fnlwgt', 'education', 'sex', 'race'])

categorical_cols = ['workclass', 'marital.status', 'occupation', 'relationship', 'native.country']
print('Encoding these categorical columns:', categorical_cols)

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print('\nShape after encoding:', X_encoded.shape)
X_encoded.head()

Encoding these categorical columns: ['workclass', 'marital.status', 'occupation', 'relationship', 'native.country']

Shape after encoding: (30162, 75)


,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,82,9,0,4356,18,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,54,4,0,3900,40,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,41,10,0,3900,40,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,34,9,0,3770,45,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,38,6,0,3770,40,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False


## Step 5: Train/Test Split

We split 80/20, stratifying on the target so both sets keep the same ~25% high-income rate. The sensitive attributes are split using the same indices so they stay aligned with the test set for the fairness audit.

In [6]:
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X_encoded, y, sensitive, test_size=0.2, random_state=42, stratify=y
)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train >50K rate: {y_train.mean():.3f}, Test >50K rate: {y_test.mean():.3f}')

Train shape: (24129, 75), Test shape: (6033, 75)
Train >50K rate: 0.249, Test >50K rate: 0.249


## Step 6: Train the Model

We use a Random Forest classifier — a good default for tabular data like this, robust to mixed feature types and less prone to overfitting than a single decision tree.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8644123984750539

              precision    recall  f1-score   support

       <=50K       0.87      0.96      0.91      4531
        >50K       0.83      0.57      0.68      1502

    accuracy                           0.86      6033
   macro avg       0.85      0.77      0.80      6033
weighted avg       0.86      0.86      0.86      6033

Confusion matrix:
[[4354  177]
 [ 641  861]]


### Why These Results? (Feature Importance)

A Random Forest can report which features it relied on most when making decisions. This gives us a concrete, evidence-based reason for the model's behavior — not just a description of it.

In [9]:
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('Top 10 most influential features:')
print(importances.head(10))

Top 10 most influential features:
capital.gain                         0.185783
marital.status_Married-civ-spouse    0.185465
education.num                        0.126249
age                                  0.076621
marital.status_Never-married         0.067777
hours.per.week                       0.062584
capital.loss                         0.045226
relationship_Not-in-family           0.040544
relationship_Own-child               0.031461
occupation_Exec-managerial           0.031349
dtype: float64


**Reading this:** `capital.gain` and `marital.status_Married-civ-spouse` are the two most influential features — together they explain more of the model's decisions than age, hours worked, or occupation. This matters for the fairness audit below: marital status and relationship category are **strongly correlated with sex** in this dataset, which is the concrete mechanism behind the group disparities we find next, not just a general possibility.

## Step 7: Fairness Audit

Our model never saw `sex` or `race` during training — but that doesn't guarantee fair outcomes. Other features (occupation, marital status, hours worked, etc.) can be correlated with sex and race, so the model can still reproduce real-world disparities indirectly. This is a well-known finding in AI fairness research often called **"fairness through unawareness" doesn't work**.

We check three metrics per group:

- **Positive Prediction Rate** (Demographic Parity): what fraction of each group gets predicted as `>50K`? If groups get very different rates, the model is not treating them equally.
- **False Negative Rate (FNR)**: among people who actually earn `>50K`, what fraction did the model wrongly predict as `<=50K`? A higher FNR for one group means real high-earners in that group are more likely to be missed.
- **False Positive Rate (FPR)**: among people who actually earn `<=50K`, what fraction did the model wrongly predict as `>50K`?

We also report per-group **accuracy** — but note that accuracy alone can be misleading when class sizes are imbalanced across groups, which is why we look at TPR/FPR/FNR separately rather than relying on accuracy alone.

In [8]:
# Combine test set sensitive attributes with true labels and predictions
audit = sens_test.reset_index(drop=True).copy()
audit['y_true'] = y_test.reset_index(drop=True)
audit['y_pred'] = y_pred

def group_metrics(df, group_col):
    results = []
    for group, g in df.groupby(group_col):
        n = len(g)
        positive_pred_rate = (g['y_pred'] == 1).mean()

        actual_pos = g[g['y_true'] == 1]
        tpr = (actual_pos['y_pred'] == 1).mean() if len(actual_pos) > 0 else float('nan')

        actual_neg = g[g['y_true'] == 0]
        fpr = (actual_neg['y_pred'] == 1).mean() if len(actual_neg) > 0 else float('nan')

        fnr = 1 - tpr if not pd.isna(tpr) else float('nan')
        acc = (g['y_true'] == g['y_pred']).mean()

        results.append({
            'group': group, 'n': n,
            'positive_pred_rate': round(positive_pred_rate, 3),
            'TPR (recall)': round(tpr, 3),
            'FPR': round(fpr, 3),
            'FNR': round(fnr, 3),
            'accuracy': round(acc, 3)
        })
    return pd.DataFrame(results)

print('=== Fairness Audit by SEX ===')
sex_audit = group_metrics(audit, 'sex')
print(sex_audit.to_string(index=False))

print()
print('=== Fairness Audit by RACE ===')
race_audit = group_metrics(audit, 'race')
print(race_audit.to_string(index=False))

=== Fairness Audit by SEX ===
 group    n  positive_pred_rate  TPR (recall)   FPR   FNR  accuracy
Female 1929               0.069         0.500 0.013 0.500     0.931
  Male 4104               0.221         0.586 0.055 0.414     0.833

=== Fairness Audit by RACE ===
             group    n  positive_pred_rate  TPR (recall)   FPR   FNR  accuracy
Amer-Indian-Eskimo   44               0.114         0.571 0.027 0.429     0.909
Asian-Pac-Islander  179               0.207         0.636 0.067 0.364     0.860
             Black  556               0.088         0.512 0.015 0.488     0.915
             Other   51               0.020         0.000 0.021 1.000     0.922
             White 5203               0.182         0.576 0.041 0.424     0.858


In [ ]:
# Confirm the proxy mechanism: relationship category vs. sex
print('Relationship by sex (raw counts):')
print(pd.crosstab(df_clean['sex'], df_clean['relationship']))

print()
married_rate = df_clean.assign(married=df_clean['marital.status']=='Married-civ-spouse').groupby('sex')['married'].mean()
print('Married-civ-spouse rate by sex:')
print((married_rate * 100).round(1))

### Reading the Results

**By sex:**
- Positive prediction rate: **22.1% for men** vs **6.9% for women** — a large demographic parity gap.
- False Negative Rate: **50% for women** vs **41.4% for men** — a genuinely high-earning woman is more likely than a genuinely high-earning man to be incorrectly predicted as low-income.
- Note that women show *higher* overall accuracy — this is misleading, since most women fall in the majority `<=50K` class, inflating accuracy while masking the recall gap.

**By race:**
- Positive prediction rates range from **2% (Other)** to **20.7% (Asian-Pac-Islander)**.
- **Black individuals** have a notably high False Negative Rate (**48.8%**) compared to White individuals (**42.4%**) — real high-earners in this group are more likely to be missed by the model.
- The **Other** group is very small in the test set (51 people), so its 100% FNR should be treated as a small-sample caveat rather than a robust conclusion.

### The Concrete Reason This Happens

This isn't just a theoretical risk — we can point to the exact mechanism. `relationship` is one of the model's top-10 features, and in this dataset it is almost a direct stand-in for sex: **12,462 of 12,463 people coded `Husband` are male, and 1,405 of 1,406 people coded `Wife` are female.** Similarly, the `Married-civ-spouse` rate is **61.8% for men vs. 15.1% for women** — and that feature is the single second-most influential one in the whole model.

So even though `sex` was never a training column, the model effectively re-derives it through `relationship` and `marital.status`, then uses that reconstructed signal when predicting income. This is the concrete, provable version of **"fairness through unawareness" doesn't work** — not a hypothetical, but something visible directly in the feature importances and the crosstab above.

**Key takeaway:** even though `sex` and `race` were excluded from training, the model still reproduces real-world income disparities almost exactly — because other features correlate with these sensitive attributes strongly enough to reconstruct the signal. Removing a sensitive attribute from the input data does not guarantee a fair outcome.